# N4 — NADE and MADE priors

In [ ]:
from pathlib import Path
import sys, yaml, numpy as np, pandas as pd, torch, matplotlib.pyplot as plt
ROOT=Path.cwd()
if not (ROOT/'src').exists(): ROOT=ROOT.parent
sys.path.insert(0,str(ROOT/'src'))
CFG=yaml.safe_load((ROOT/'config.yaml').read_text())
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
print('device:',DEVICE,'debug:',CFG['data']['debug'])

from torch.utils.data import DataLoader,TensorDataset
from piano_transcriber.priors import NADE,MADE,nade_sample,made_sample
from piano_transcriber.checkpoints import save_checkpoint\n# Sampling sanity checks: inspect generated pitch occupancy.\nsn=nade_sample(nade,128,DEVICE).cpu().numpy(); sm=made_sample(made,128,DEVICE).cpu().numpy()\nprint('NADE / MADE sample shapes:',sn.shape,sm.shape)\nprint('Mean active pitches:',sn.sum(1).mean(),sm.sum(1).mean())\n

Replace the smoke-test Bernoulli samples with the training/validation piano-roll frame tensors exported by N1. Exact NLL is computed by the autoregressive factorization.

In [ ]:
x=torch.bernoulli(torch.full((4096,88),.08));v=torch.bernoulli(torch.full((512,88),.08))
loader=DataLoader(TensorDataset(x),batch_size=128,shuffle=True)
nade=NADE(88,CFG['prior']['hidden_units']).to(DEVICE);made=MADE(88,(256,256)).to(DEVICE)
def fit(model):
 opt=torch.optim.Adam(model.parameters(),lr=CFG['prior']['learning_rate'])
 for ep in range(CFG['prior']['epochs']):
  model.train();ls=[]
  for (b,) in loader:
   b=b.to(DEVICE);opt.zero_grad();loss=model(b);loss.backward();opt.step();ls.append(loss.item())
  if (ep+1)%5==0:print(ep+1,np.mean(ls))
fit(nade);fit(made)
save_checkpoint(ROOT/'models/nade.pt',nade,{'n_visible':88,'n_hidden':CFG['prior']['hidden_units']})
save_checkpoint(ROOT/'models/made.pt',made,{'n_visible':88,'hidden_sizes':[256,256]})

In [ ]:
nade.eval();made.eval()
with torch.no_grad():
 print('NADE validation NLL:',-nade.log_prob(v.to(DEVICE)).mean().item())
 print('MADE validation NLL:',-made.log_prob(v.to(DEVICE)).mean().item())

Sampling checks: implement ancestral sampling in the same ordering/mask scheme and compare pitch occupancy and chord-size histograms against held-out frames. Repeat with multiple orderings/seeds.